# Laboratorio 2: Preprocesamiento de Datos
Este archivo `.ipynb` contiene el desarrollo estructurado de las tareas solicitadas para el laboratorio sobre manipulación y limpieza de datos empleando la librería **pandas** y **numpy**.

### Objetivos:
1. Manejar datos perdidos de diferentes maneras.
2. Corregir el tipo de datos de diferentes valores según los requisitos.
3. Estandarizar y normalizar de manera apropiada los datos numéricos.
4. Visualizar los datos como un gráfico de barras agrupadas (Binning).
5. Convertir datos categóricos a numéricos (One-Hot Encoding).


## Inicialización y Carga del Dataset
Se cargan las librerías iniciales necesarias y se procede a inspeccionar estructuralmente el dataset `laptop_pricing_dataset_mod1.csv` usando los métodos `.head()` y `.info()`.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Cargar el dataset
try:
    df = pd.read_csv('laptop_pricing_dataset_mod1.csv')
    print('Dataset cargado con éxito.')
except FileNotFoundError:
    print('Error: Asegúrate de colocar el archivo "laptop_pricing_dataset_mod1.csv" en el mismo directorio.')

# Mostrar información preliminar
print('\n--- Primeras 5 filas del Dataset ---')
print(df.head())

print('\n--- Estructura e Información del Dataset ---')
print(df.info())


## Tarea 1: Tratamiento de Valores Faltantes (Missing Values)
Identificación Crítica: Algunos valores faltantes no se presentan como nulos tradicionales (`NaN`), sino que están representados por caracteres especiales como `?`. Usamos el método `.replace()` de pandas para estandarizarlos.


In [ ]:
# Reemplazar '?' por NaN estándar
df.replace('?', np.nan, inplace=True)
print('Valores "?" sustituidos satisfactoriamente por NaN.')
print('Cantidad de nulos detectados por columna ahora:')
print(df.isnull().sum())


## Tarea 2: Imputación de Valores Faltantes
- Los valores faltantes en atributos continuos (`Peso_kg`) se reemplazan con el valor promedio (media).
- Los valores faltantes en atributos categóricos (`Screen_Size_cm`) se reemplazan con el valor más frecuente (la moda).


In [ ]:
# Reemplazar valores faltantes en 'Peso_kg' con la media
average_peso = df['Peso_kg'].astype('float').mean(axis=0)
df['Peso_kg'].replace(np.nan, average_peso, inplace=True)

# Reemplazar valores faltantes en 'Screen_Size_cm' con el valor más frecuente (moda)
most_frequent_screen = df['Screen_Size_cm'].value_counts().idxmax()
df['Screen_Size_cm'].replace(np.nan, most_frequent_screen, inplace=True)

print(f'Imputación completada.')
print(f' - Valor medio asignado a Peso_kg: {average_peso:.3f}')
print(f' - Valor más frecuente asignado a Screen_Size_cm: {most_frequent_screen}')


## Tarea 3: Corregir los Tipos de Datos
Tanto `Peso_kg` como `Screen_Size_cm` inicialmente se cargan como tipo `Object` debido a los caracteres especiales. Tras la limpieza, corregimos su tipo de datos a flotante (`float`).


In [ ]:
# Corregir el tipo de dato a flotante
df[['Peso_kg', 'Screen_Size_cm']] = df[['Peso_kg', 'Screen_Size_cm']].astype('float')
print('Tipos de datos corregidos con éxito:')
print(df[['Peso_kg', 'Screen_Size_cm']].dtypes)


## Tarea 4: Estandarización de Datos y Normalización
- Convertir el tamaño de pantalla a pulgadas (1 pulgada = 2.54 cm).
- Convertir el peso de la laptop a libras (1 kg = 2.205 libras).
- Renombrar las columnas de manera acorde.
- Normalizar el atributo continuo `CPU_frequency` respecto a su valor máximo.


In [ ]:
# Conversión de unidades
df['Screen_Size_cm'] = df['Screen_Size_cm'] / 2.54
df['Peso_kg'] = df['Peso_kg'] * 2.205

# Actualizar nombres de las columnas
df.rename(columns={'Screen_Size_cm': 'Screen_Size_inch', 'Peso_kg': 'Peso_lbs'}, inplace=True)

# Normalizar CPU_frequency respecto al valor máximo
df['CPU_frequency'] = df['CPU_frequency'] / df['CPU_frequency'].max()

print('Estandarización y Normalización finalizada. Vista previa:')
print(df[['Screen_Size_inch', 'Peso_lbs', 'CPU_frequency']].head())


## Tarea 5: Agrupamiento (Binning)
Creamos un atributo categórico discreto dividiendo los valores continuos de la columna `Precio` en 3 grupos: `Bajo`, `Medio` y `Alto`. El nuevo atributo se guardará bajo el nombre `Agrupamiento de precios` y se visualizará gráficamente.


In [ ]:
# Definir los límites del agrupamiento lineal entre el mínimo y máximo
bins = np.linspace(min(df['Precio']), max(df['Precio']), 4)
group_names = ['Bajo', 'Medio', 'Alto']

# Aplicar el corte (Binning)
df['Agrupamiento de precios'] = pd.cut(df['Precio'], bins, labels=group_names, include_lowest=True)

# Dibujar el gráfico de barras
plt.figure(figsize=(8, 5))
df['Agrupamiento de precios'].value_counts().reindex(group_names).plot(kind='bar', color=['#4C72B0', '#55A868', '#C44E52'], edgecolor='black')
plt.xlabel('Categoría de Precio', fontsize=12)
plt.ylabel('Cantidad de Laptops', fontsize=12)
plt.title('Gráfico de Barras - Agrupamiento de Precios', fontsize=14)
plt.xticks(rotation=0)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()


## Tarea 6: Variables Ficticias o Indicadoras (One-Hot Encoding)
Convertimos el atributo categórico `Pantalla` en dos variables numéricas indicadoras individuales. Tras concatenarlas al dataframe principal, removemos la columna original.


In [ ]:
# Crear variables indicadoras para la columna 'Pantalla'
dummy_variable = pd.get_dummies(df['Pantalla'])

# Intentar mapear y renombrar dinámicamente las columnas dummy de acuerdo al requerimiento
# Se asume que los valores contienen variantes de 'IPS panel' y 'Full HD'
rename_dict = {}
for col in dummy_variable.columns:
    if 'IPS' in col or 'ips' in col.upper():
        rename_dict[col] = 'Pantalla-IPS_panel'
    elif 'HD' in col or 'hd' in col.upper():
        rename_dict[col] = 'Pantalla-Full_HD'
    else:
        rename_dict[col] = f'Pantalla-{col.replace(" ", "_")}'

dummy_variable.rename(columns=rename_dict, inplace=True)

# Concatenar e integrar las nuevas variables dummy al dataframe
df = pd.concat([df, dummy_variable], axis=1)

# Eliminar el atributo original 'Pantalla'
if 'Pantalla' in df.columns:
    df.drop('Pantalla', axis=1, inplace=True)

print('One-Hot Encoding completado con éxito. Columnas actuales en el DataFrame:')
print(df.columns.tolist())
